# Shared Data Utilities — Vibration Signals

Import the libraries needed for reproducibility, tensor conversion, and DataLoader creation.

These functions will be reused across the vibration-based fault diagnosis models.

In [1]:
import random
import numpy as np
import torch

from torch.utils.data import TensorDataset, DataLoader

## 1. Reproducibility

Define a function to set the random seed for Python, NumPy, and PyTorch.

This helps make the experiments reproducible and allows the models to be evaluated using the same set of random seeds.

In [2]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 2. Vibration Dataset Preparation

Convert vibration signals and their corresponding fault labels into PyTorch tensors.

The function adds the channel dimension required by 1-D convolutional models and checks that the signals and labels have compatible dimensions.

It supports different vibration window sizes without requiring changes to the function.

In [3]:
def make_dataset(X, y):

    X = torch.as_tensor(X, dtype=torch.float32)

    if X.ndim == 2:
        X = X.unsqueeze(1)

    if X.ndim != 3 or X.shape[1] != 1:
        raise ValueError(
            "Expected vibration signals with shape (N, L) or (N, 1, L)."
        )

    y = torch.as_tensor(y, dtype=torch.long)

    if y.ndim != 1 or len(X) != len(y):
        raise ValueError(
            "Labels must be a 1-D array matching the number of signals."
        )

    return TensorDataset(X, y)

## 3. DataLoader Creation

Define a reusable function to create PyTorch DataLoaders with a configurable batch size and shuffling option.

Training data can be shuffled, while validation and test data are kept in a fixed order.

In [4]:
def make_loader(dataset, batch_size=256, shuffle=False):

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

## 4. Domain-Generalization Dataset Preparation

Load a selected domain-generalization task using the preprocessing function.

Extract only the vibration signals and fault labels from the training, validation, and unseen target-domain test sets, and convert them into PyTorch datasets.

This function can be reused for Tasks 9–12 by changing the target domain.

In [5]:
def create_task_datasets(data_path, target_domain, load_dg_task):

    task = load_dg_task(
        str(data_path),
        target_domain=target_domain
    )

    train_dataset = make_dataset(
        task["train"][0],
        task["train"][2]
    )

    val_dataset = make_dataset(
        task["val"][0],
        task["val"][2]
    )

    test_dataset = make_dataset(
        task["target"][0],
        task["target"][2]
    )

    return train_dataset, val_dataset, test_dataset

## 5. Domain-Generalization DataLoaders

Create the training, validation, and test DataLoaders for a selected domain-generalization task.

The function combines dataset preparation and DataLoader creation, using shuffling only for training.

This avoids repeating the same data preparation code in the CNN and ResNet notebooks.

In [6]:
def create_task_loaders(
    data_path,
    target_domain,
    load_dg_task,
    batch_size=256
):

    train_dataset, val_dataset, test_dataset = create_task_datasets(
        data_path,
        target_domain,
        load_dg_task
    )

    train_loader = make_loader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = make_loader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    test_loader = make_loader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, val_loader, test_loader

## 6. Utility Function Verification

Test the shared functions using a small synthetic vibration dataset.

Verify that the signals have the expected channel dimension, that labels use the correct data type, and that the DataLoader produces batches of the requested size.

In [7]:
set_seed(0)

X_sample = np.random.randn(100, 1024).astype(np.float32)
y_sample = np.random.randint(0, 6, size=100)

sample_dataset = make_dataset(X_sample, y_sample)

sample_loader = make_loader(
    sample_dataset,
    batch_size=32,
    shuffle=True
)

xb, yb = next(iter(sample_loader))

print("Vibration batch:", xb.shape)
print("Labels batch:", yb.shape)
print("Labels dtype:", yb.dtype)

Vibration batch: torch.Size([32, 1, 1024])
Labels batch: torch.Size([32])
Labels dtype: torch.int64
